# Bases de la Computación de Alto Rendimiento (HPC)

En la notebook anterior viste que la *forma* de escribir un cálculo puede cambiar su velocidad 100 veces o más. Esta notebook explica **por qué**: qué hace que una computadora sea rápida o lenta, y qué reglas básicas debes tener en cuenta al escribir código que necesita rendimiento.

A lo largo de la notebook vas a alternar entre teoría y actividades cortas para medir estas ideas en tu propia laptop. No necesitas un clúster para entender los conceptos: los mismos principios (memoria, cachés, cuellos de botella) existen, a menor escala, dentro de tu computadora.

### Contenido
* ¿Qué es HPC?
* Clústeres HPC: hardware
* La CPU y las operaciones de punto flotante (FLOPS)
* Acceso a memoria
* Entrada/salida (I/O)

## ¿Qué es HPC?

Una definición útil (Georg Hager, RRZE): **HPC es programar teniendo en cuenta los cuellos de botella del hardware.**

Es decir: escribir *software eficiente* que use el *máximo potencial del hardware*. Para lograrlo hace falta:
* Entender el hardware (¿qué partes de la computadora existen y cómo se relacionan?).
* Entender cómo el software se ejecuta sobre ese hardware (¿qué hace lento a un programa?).

Para dimensionar de qué estamos hablando: tu laptop probablemente puede hacer del orden de decenas de miles de millones de operaciones por segundo (varias decenas de GFLOPS). Un supercomputador de los más grandes del mundo hoy hace más de mil millones de millones de operaciones por segundo (varios ExaFLOPS) — es decir, del orden de un millón de veces más que tu laptop. Pero ese poder no sirve de nada si el programa está mal escrito: un código ineficiente en un supercomputador puede llegar a rendir peor, en proporción, que un código bien escrito en tu laptop. Ese es justamente el problema que resuelve HPC: **no solo tener hardware potente, sino saber usarlo**.

Los cuellos de botella más comunes son:
* **Operaciones de punto flotante** (cálculos numéricos) — ¿cuántas puede hacer la CPU por segundo?
* **Acceso a memoria** — ¿qué tan rápido se puede leer/escribir un dato?
* **Entrada/salida (I/O)** — ¿qué tan rápido se puede leer/escribir un archivo en disco?
* **Comunicación** — cuando varios procesos necesitan hablar entre sí (lo verás en el notebook de MPI).

Un **cuello de botella** es, literalmente, el paso más lento de una cadena: no importa qué tan rápidos sean los demás pasos, el más lento determina el tiempo total. Identificar cuál de estos cuatro es el cuello de botella de *tu* programa específico es el primer paso antes de intentar optimizarlo — optimizar el paso equivocado no mejora nada.

Esta notebook cubre solo lo esencial de cada uno. Si te interesa profundizar en HPC "de bajo nivel", existen cursos dedicados (HLRS, LRZ, RRZE, ver notebook 01).

## Hardware de un clúster HPC

Un clúster típico de HPC tiene estos componentes:

![cluster](fig/cluster_components.svg)

* **Login node (nodo de acceso):** por aquí te conectas tú; se usa para preparar y enviar trabajos, no para calcular.
* **Compute nodes (nodos de cómputo):** aquí es donde realmente corren tus simulaciones. Hay muchos ("More nodes...").
* **Network (red):** conecta todos los nodos entre sí a alta velocidad.
* **Storage (almacenamiento):** donde viven tus archivos y datos, accesible desde todos los nodos.

La idea clave: en vez de una sola computadora, tienes cientos o miles de "compute nodes" trabajando a la vez, conectados por una red rápida. Tu laptop, en esta analogía, es como un único "compute node" — todo lo que veas en el resto de esta notebook (CPU, memoria, I/O) aplica igual dentro de un nodo del clúster que dentro de tu laptop.

## Paralelismo en un clúster HPC

<img src="fig/cluster_parallelism.svg" style="float:right; width:35%" />

El paralelismo existe en varios niveles, uno dentro de otro, como muñecas rusas:

* **Clúster → nodos:** muchas computadoras separadas, conectadas por red.
* **Nodo → sockets (CPUs), GPUs:** cada nodo puede tener más de un CPU físico, y a veces una o varias GPUs.
* **CPU → cores (núcleos):** cada CPU tiene varios núcleos que pueden calcular cosas distintas al mismo tiempo.
* **Core → SIMD (Single Instruction, Multiple Data):** incluso dentro de un solo núcleo, el hardware puede aplicar la misma operación a varios números a la vez.

Vas a explorar cómo aprovechar estos niveles de paralelismo más adelante en el curso (multiprocessing, Dask, MPI, GPU). Por ahora, quédate con la idea de que "más paralelo" no es solo "más computadoras": también es más núcleos dentro de un mismo chip, y más operaciones simultáneas dentro de un mismo núcleo.

### Actividad: explora el hardware de tu propia laptop

Tu laptop ya tiene varios de estos niveles de paralelismo, aunque a menor escala que un clúster. Ejecuta la siguiente celda para ver cuántos núcleos "ve" Python en tu máquina.

In [1]:
import os

n_logicos = os.cpu_count()
print(f"Nucleos logicos disponibles: {n_logicos}")

Nucleos logicos disponibles: 12


**Nota sobre "núcleos lógicos":** muchos procesadores usan una técnica llamada *hyperthreading* (o SMT), que hace que cada núcleo físico se presente ante el sistema operativo como 2 núcleos "lógicos". Es decir, si tu laptop tiene 4 núcleos físicos, es posible que `os.cpu_count()` te devuelva 8. No duplica la capacidad de cálculo real, pero sí ayuda a mantener el núcleo ocupado mientras espera datos de memoria (justo el problema que vas a ver en la sección de memoria más abajo).

**Tus resultados**

- Núcleos lógicos que reportó tu laptop: ___
- Busca en internet el modelo exacto de tu CPU (o revisa la configuración de tu sistema) y anota cuántos núcleos *físicos* tiene: ___
- ¿Coincide `os.cpu_count()` con el número de núcleos físicos, o es el doble? ¿Por qué crees que pasa eso?

## La CPU (unidad central de procesamiento)

<img src="fig/cpu.svg" style="float:right; width:35%" />

Versión simplificada de cómo funciona una CPU:

* **Front end** ("entrada"): obtiene y decodifica las instrucciones — lee tu programa compilado y entiende qué hay que hacer.
* **Back end** ("salida"): ejecuta esas instrucciones sobre los datos, y mueve datos entre los registros (memoria interna, minúscula pero instantánea) y la memoria principal (RAM).
* La CPU opera a una frecuencia determinada (los GHz de la ficha técnica) y puede procesar un número limitado de operaciones por segundo.

**Posible cuello de botella:** el número de operaciones de punto flotante por segundo que la CPU puede sostener, medido en **FLOPS** (*Floating Point Operations Per Second*). Si tu código ya exprime al máximo la capacidad de cálculo de la CPU, no vas a lograr que corra más rápido sin cambiar de hardware — pero, como vimos en el teaser, la mayoría del código en la vida real ni siquiera se acerca a ese límite: el cuello de botella suele estar en otro lado (memoria, por ejemplo).

### Actividad: mide cuántos FLOPS logra tu laptop

Vamos a comparar dos formas de multiplicar matrices — la misma operación matemática, para conectar esta actividad directamente con lo que viste en el teaser de difusión:

1. Con loops explícitos en Python puro.
2. Con el operador `@` de NumPy (multiplicación de matrices optimizada, usando una librería BLAS escrita en C/Fortran).

Una multiplicación de matrices de tamaño N×N requiere aproximadamente $2N^3$ operaciones de punto flotante (por cada elemento del resultado, se hacen N multiplicaciones y N sumas). Con eso podemos estimar cuántos FLOPS logra cada versión.

**Instrucciones:** ejecuta las dos celdas siguientes. Usamos un tamaño de matriz pequeño (`N=80`) para la versión en loops puros, porque si no tardaría demasiado — ese ya es un primer indicio de la diferencia de rendimiento.

In [2]:
import numpy as np
import time

def flops_loops(N):
    A = np.random.rand(N, N)
    B = np.random.rand(N, N)
    C = np.zeros((N, N))
    inicio = time.time()
    for i in range(N):
        for j in range(N):
            suma = 0.0
            for k in range(N):
                suma += A[i, k] * B[k, j]
            C[i, j] = suma
    tiempo = time.time() - inicio
    flops = 2 * N**3
    gflops = flops / tiempo / 1e9
    return tiempo, gflops

N = 80
tiempo, gflops = flops_loops(N)
print(f"Loops puros  (N={N}): {tiempo:.3f} s -> {gflops:.4f} GFLOPS")

Loops puros  (N=80): 0.113 s -> 0.0091 GFLOPS


In [3]:
def flops_numpy(N):
    A = np.random.rand(N, N)
    B = np.random.rand(N, N)
    inicio = time.time()
    C = A @ B
    tiempo = time.time() - inicio
    flops = 2 * N**3
    gflops = flops / tiempo / 1e9
    return tiempo, gflops

# usamos un N mucho mas grande porque esta version es mucho mas rapida
N = 800
tiempo, gflops = flops_numpy(N)
print(f"NumPy (@)    (N={N}): {tiempo:.3f} s -> {gflops:.2f} GFLOPS")

NumPy (@)    (N=800): 0.016 s -> 65.01 GFLOPS


**Tus resultados**

- GFLOPS con loops puros: ___
- GFLOPS con NumPy: ___
- ¿Cuántas veces más GFLOPS logró la versión de NumPy?

La versión con `@` no es más rápida porque "sepa multiplicar matrices mejor" — hace exactamente las mismas multiplicaciones y sumas. Es más rápida porque usa una librería (BLAS) escrita en C/Fortran, compilada específicamente para tu CPU, que aprovecha SIMD (varias operaciones por instrucción) y organiza el acceso a memoria para aprovechar la caché — todo lo que vas a ver en la siguiente sección. La versión en loops puros, en cambio, ni siquiera se acerca al límite físico de FLOPS de tu CPU: el cuello de botella ahí no es la CPU, es el overhead del intérprete de Python.

## Acceso a memoria

<img src="fig/memory_hierarchy.svg" style="float:right; width:35%" />

Un programa de HPC necesita datos para trabajar, y esos datos tienen que llegar hasta la CPU. El problema: **la memoria (RAM) es mucho más lenta que la CPU**. Si cada cálculo tuviera que esperar a que llegue un dato desde la RAM, la CPU pasaría la mayor parte del tiempo esperando, no calculando.

La solución es la **jerarquía de memoria**: capas de memoria cada vez más rápidas (pero más pequeñas) cerca de la CPU:

* **Registros:** minúsculos, dentro de la CPU, velocidad casi instantánea.
* **Caché L1, L2, L3:** cada vez más grandes pero más lentos, van "adelantando" datos de la RAM antes de que la CPU los pida.
* **Memoria principal (RAM):** grande, pero mucho más lenta que la caché.

Piénsalo como una biblioteca: los **registros** son los libros que tienes abiertos en tu escritorio ahora mismo; la **caché** es el estante a tu lado con los libros que probablemente vas a necesitar pronto; la **RAM** es el depósito general del edificio, al que tienes que caminar si el libro no está ni en tu escritorio ni en el estante. Cada vez que tienes que ir hasta el depósito (RAM) en vez de estirar la mano al estante (caché), pierdes muchísimo tiempo en proporción al tiempo que toma leer el libro en sí.

Cuando la CPU pide un dato, la caché no trae solo ese dato: trae un bloque completo de datos vecinos (una "línea de caché"), apostando a que vas a necesitar esos datos vecinos pronto. Si tu programa efectivamente usa esos vecinos a continuación, "acierta" en caché (rápido); si no, tiene que ir hasta la RAM de nuevo (lento). A esto se le llama localidad de acceso.

De ahí salen dos reglas prácticas muy importantes:
* **Accede a la memoria de forma lineal** (en el mismo orden en que está almacenada), para aprovechar cada línea de caché que se trae.
* **Reutiliza los datos que ya están en caché** tanto como sea posible, en vez de traer datos nuevos todo el tiempo.

### Arreglos multidimensionales: ¿cómo se guardan en memoria?

<img src="fig/row-column-major.svg" style="float:right; width: 25%" />

La memoria de la computadora es, en el fondo, una sola fila larga de casillas (1D). Entonces, ¿cómo se guarda un arreglo de 2 dimensiones (como una matriz o la grilla del teaser)? Hay dos convenciones:

* **Column-major (por columnas):** se guarda una columna completa, después la siguiente. La usa Fortran.
* **Row-major (por filas):** se guarda una fila completa, después la siguiente. La usan C y Python/NumPy.

Esto importa porque, si accedes a los datos en un orden distinto al que están guardados, "saltas" por la memoria en vez de leerla de forma lineal — y pierdes toda la ventaja de la caché que acabamos de explicar.

**Regla práctica para loops anidados:** el índice que cambia más rápido en tu loop más interno debe coincidir con cómo está guardada la memoria:
* Column-major (Fortran): el índice que más rápido cambia debe ser el **primero**.
* Row-major (C, Python/NumPy): el índice que más rápido cambia debe ser el **último**.

### Ejemplo en Fortran (column-major)

```fortran
double precision :: a(N, M)
integer :: i, j
do j = 1, M
  do i = 1, N
    a(i, j) = i*j
  end do
end do
```

Aquí `i` (el primer índice) cambia en el loop más interno — coincide con cómo Fortran guarda la matriz.

### Ejemplo en C (row-major)

```c
double a[N][M];
int i, j;
for (i = 0; i < N; i++)
  for (j = 0; j < M; j++)
    a[i][j] = i*j;
```

Aquí `j` (el último índice) cambia en el loop más interno — coincide con cómo C guarda la matriz.

## Actividad: ¿se nota la diferencia en Python puro?

Python y NumPy usan row-major, igual que C. Vamos a comparar llenar un arreglo en el orden "correcto" (`i` afuera, `j` adentro) contra el orden "incorrecto" (`j` afuera, `i` adentro).

**Instrucciones:** ejecuta la siguiente celda. `%timeit` corre la función varias veces automáticamente y te da un tiempo promedio — no hace falta usar `%%time` a mano.

In [4]:
import numpy as np
N = 1000
array = np.zeros((N, N))

def set_array_row_major():
    # orden "correcto" para row-major: i (fila) afuera, j (columna) adentro
    for i in range(N):
        for j in range(N):
            array[i, j] = i*j

def set_array_column_major():
    # orden "incorrecto" para row-major: j afuera, i adentro
    for j in range(N):
        for i in range(N):
            array[i, j] = i*j

%timeit set_array_row_major()
%timeit set_array_column_major()

94.5 ms ± 2.12 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
96 ms ± 1.92 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Tus resultados**

- Tiempo con orden correcto (row-major): _94.1__
- Tiempo con orden incorrecto (column-major): _94.8__
- ¿Hubo una diferencia grande? _No, la diferencia es prácticamente nula__

Es probable que la diferencia sea pequeña, o casi imperceptible. Esto pasa porque, en Python puro, cada iteración del loop ya carga con un overhead grande (Python tiene que interpretar cada línea, revisar tipos, etc.), del orden de decenas de nanosegundos por operación. El costo *extra* de fallar la caché por acceder en el orden equivocado es, en comparación, mucho más chico. Es un cuello de botella grande (el intérprete) tapando a uno más chico (la memoria) — como tratar de escuchar un susurro al lado de una bocina.

Para ver el efecto real, necesitamos quitar ese overhead compilando el código, para que el único cuello de botella que quede sea la memoria. Vamos a usar **Cython**, que compila una versión anotada de Python a C.

**Nota:** esta parte necesita un compilador de C instalado (el mismo que configuraste en la Tarea 1 del notebook 01, con `environment.yml` o `environment2.yml`). Si te da error al ejecutar la celda `%%cython`, probablemente falta el compilador — revisa esa tarea antes de continuar.

In [5]:
%pip install cython

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install setuptools

Note: you may need to restart the kernel to use updated packages.


In [7]:
%load_ext Cython

In [8]:
%%cython -c=-O3 -c=-march=native
import numpy as np
cimport numpy as np
cimport cython

N = 10000

@cython.boundscheck(False) # desactiva la verificacion de limites del arreglo
@cython.wraparound(False)  # desactiva los indices negativos (array[-1])
def set_array_row_major_cython():
    cdef int i, j
    cdef np.ndarray[np.float64_t, ndim=2] array = np.zeros((N, N), dtype=np.float64)
    for i in range(N):
        for j in range(N):
            array[i, j] = i*j

@cython.boundscheck(False)
@cython.wraparound(False)
def set_array_column_major_cython():
    cdef int i, j
    cdef np.ndarray[np.float64_t, ndim=2] array = np.zeros((N, N), dtype=np.float64)
    for j in range(N):
        for i in range(N):
            array[i, j] = i*j

In [9]:
%timeit set_array_row_major_cython()
%timeit set_array_column_major_cython()

256 ms ± 132 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
357 ms ± 6.56 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


##### **Tus resultados (versión Cython, sin overhead de Python)**

- Tiempo con orden correcto (row-major): 256 ms___
- Tiempo con orden incorrecto (column-major): 357 ms___
- Esta vez, ¿la diferencia fue más notoria que en la versión de Python puro? ¿Por qué crees que pasó eso, después de leer la explicación de la jerarquía de memoria de arriba?
  
Sí, bastante más notoria (fueron como 100 ms de diferencia). Pasó porque Cython le quitó a Python toda la lentitud de interpretar el código, dejando al descubierto el cuello de botella de la memoria.

## Entrada y salida (I/O)

Los sistemas de HPC normalmente usan **sistemas de archivos paralelos**, diseñados para que muchos núcleos puedan leer y escribir datos grandes al mismo tiempo sin saturarse.

* El objetivo es I/O eficiente para grandes volúmenes de datos escritos por muchos procesos a la vez.
* Los archivos se distribuyen entre muchos discos.
* Componentes típicos:
  * **Servidores de almacenamiento:** guardan el contenido real de los archivos.
  * **Servidores de metadatos:** guardan información sobre dónde está cada archivo.
* Acceder a un archivo implica dos pasos:
  1. Preguntarle al servidor de metadatos **dónde** está guardado el archivo.
  2. Pedirle los datos al servidor de almacenamiento correspondiente.
* Esto se repite para cada proceso, para cada archivo que abre.
* Ejemplos de este tipo de sistemas: GPFS, Lustre, BeeGFS.

### Buenas prácticas de I/O

* Evita hacer I/O si no es estrictamente necesario.
* El servidor de metadatos suele ser el cuello de botella, no el de almacenamiento — por eso:
* **Escribe pocos archivos grandes.** Evita escribir muchos archivos pequeños (por ejemplo, miles de archivos de resultados en vez de uno solo con todo).
* Si necesitas leer un archivo pequeño (por ejemplo, un archivo de configuración) desde muchos procesos a la vez, no lo leas por separado en cada uno: **léelo una sola vez en un proceso y distribúyelo** a los demás.

### Actividad: muchos archivos pequeños vs. un archivo grande

Vas a comprobar el efecto de "muchos archivos pequeños" incluso en el disco de tu propia laptop, sin necesidad de un clúster. La idea: escribir la misma cantidad total de datos de dos formas distintas, y comparar el tiempo.

**Instrucciones:** ejecuta la celda. Al final se borran los archivos temporales que se crean para la prueba.

In [10]:
import time, os, tempfile, shutil

n_bloques = 3000
contenido = "x" * 200   # un bloque pequeno de datos

# Version 1: muchos archivos pequenos
carpeta_temp = tempfile.mkdtemp()
inicio = time.time()
for i in range(n_bloques):
    ruta = os.path.join(carpeta_temp, f"archivo_{i}.txt")
    with open(ruta, "w") as f:
        f.write(contenido)
tiempo_muchos_archivos = time.time() - inicio
shutil.rmtree(carpeta_temp)

# Version 2: un solo archivo grande con el mismo total de datos
ruta_grande = os.path.join(tempfile.gettempdir(), "archivo_grande.txt")
inicio = time.time()
with open(ruta_grande, "w") as f:
    for i in range(n_bloques):
        f.write(contenido)
tiempo_un_archivo = time.time() - inicio
os.remove(ruta_grande)

print(f"Muchos archivos pequenos ({n_bloques} archivos): {tiempo_muchos_archivos:.4f} s")
print(f"Un solo archivo grande   ({n_bloques} bloques) : {tiempo_un_archivo:.4f} s")
print(f"El primero fue {tiempo_muchos_archivos / tiempo_un_archivo:.1f} veces mas lento")

Muchos archivos pequenos (3000 archivos): 0.1408 s
Un solo archivo grande   (3000 bloques) : 0.0011 s
El primero fue 124.9 veces mas lento


**Tus resultados**

- Tiempo con muchos archivos pequeños: _0.1408s__
- Tiempo con un archivo grande: _0.0011s__
- ¿Cuántas veces más lenta fue la versión de muchos archivos?  Fue 124.9 veces más lenta.

Ese costo extra viene de que cada `open()`/`close()` tiene que hablar con el sistema de archivos para crear una entrada nueva (el equivalente, a pequeña escala, de consultar al "servidor de metadatos"). En tu laptop ese costo ya es medible; en un clúster HPC, donde ese servidor de metadatos es compartido por miles de procesos al mismo tiempo y se accede por red, el mismo patrón de "muchos archivos pequeños" puede volverse un cuello de botella severo para todo el sistema, no solo para tu programa.

Ejemplo concreto: si tu simulación genera un resultado por cada una de 10 000 combinaciones de parámetros, guardarlas en 10 000 archivos `.csv` separados va a ser mucho más lento (y generará más carga en el servidor de metadatos) que guardarlas todas juntas en un solo archivo HDF5 — justo el formato que vas a ver en el notebook de NumPy/SciPy/H5Py.

## Resumen: ¿de qué se trata HPC?

* Escribir programas que usen el hardware al límite de su capacidad.
* Tener presentes los cuellos de botella típicos: FLOPS, acceso a memoria, I/O, comunicación.
* Dos reglas básicas para quedarte con esta notebook:
  * **Accede a la memoria de forma lineal.**
  * **Escribe pocos archivos, pero grandes.**

En las siguientes notebooks vas a ver cómo NumPy, SciPy, Numba y otras herramientas te ayudan a seguir estas reglas sin tener que escribir C o Fortran a mano.

## Autoevaluación

Antes de pasar a la siguiente notebook, responde estas preguntas con tus propias palabras (edita esta celda). No se trata de repetir definiciones de memoria, sino de explicar el "por qué" — si puedes responderlas sin volver a leer arriba, ya entendiste lo esencial de esta notebook.

**¿Qué es un cuello de botella, y cuáles son los cuatro que vimos en esta notebook?**
   Un cuello de botella es el componente del sistema (ya sea de software o hardware) que frena todo el proceso por ser la parte más lenta. Los 4 que vimos son:
   * **El intérprete de Python:** La lentitud al revisar tipos e instrucciones línea por línea en loops puros.
   * **La CPU / Cómputo (FLOPS):** El límite físico de cuántas operaciones matemáticas puede procesar el chip por segundo.
   * **La memoria RAM:** La lentitud de traer datos lejanos si no se aprovecha la caché.
   * **Entrada/Salida y Metadatos:** El tiempo que pierde el sistema operativo al abrir, crear y cerrar archivos en disco.

2. **En la actividad de FLOPS, ¿por qué la versión con NumPy logró muchos más GFLOPS que la de loops puros, si ambas hacen exactamente las mismas cuentas?**
   Porque NumPy ejecuta el cálculo usando librerías compiladas en C y Fortran optimizadas a bajo nivel. Esto le permite aprovechar instrucciones SIMD del procesador (procesar varios datos al mismo tiempo) y eliminar todo el overhead del intérprete de Python, que en los loops puros pierde la mayor parte del tiempo analizando tipos de datos e instrucciones en cada iteración.

3. **¿Por qué existe la caché si ya existe la RAM?**
   Existe porque la memoria RAM es demasiado lenta en comparación con la velocidad de la CPU. Si la CPU le pidiera todo directamente a la RAM, pasaría la mayor parte del tiempo esperando datos parada sin hacer nada. La caché es una memoria diminuta pero ultra rápida pegada al procesador que guarda por adelantado los datos que se van a usar inmediatamente.

4. **Si escribes un loop anidado sobre un arreglo de NumPy, ¿qué índice conviene que cambie en el loop más interno? ¿Por qué?**
 Esto se debe a que NumPy guarda las matrices en memoria por filas (*row-major*). Al cambiar el último índice en el loop interno, recorremos la memoria de forma lineal y contigua, lo que permite que la caché guarde los datos vecinos y la ejecución sea rápida.

5. **En la actividad de archivos, ¿por qué escribir muchos archivos pequeños fue más lento que escribir un solo archivo grande con la misma cantidad de datos?**
   Porque cada vez que se crea o abre un archivo, el sistema operativo tiene que hacer un trabajo administrativo (consultar metadatos, permisos, rutas y asignar bloques). Crear miles de archivos pequeños repite esa "burocracia" miles de veces, mientras que en un archivo grande solo se hace esa gestión una vez y luego los datos se escriben de corrido.